# Conteo de registros H y M

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 1–11.

Versión para completar. Puedes trabajar en este cuaderno en lugar de `alumno.py`. Sustituye los `NotImplementedError` por tu código. Las celdas con tareas pendientes se detendrán hasta que las completes.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Conteo de registros por categoría H/M

El objetivo consiste en contar cuántos registros contienen la etiqueta H y cuántos la etiqueta M. La categoría se lee del último campo. El nombre no determina esa etiqueta.

Trabaja en `alumno.py`. Necesitas Python 3.10 o posterior y su biblioteca estándar. Los imports y los datos ya están preparados. Sustituye cada `NotImplementedError` por tu implementación. Ver ese error antes de completar una función es normal en esta plantilla.

## Datos y alcance

Hay tres ficheros con tres registros cada uno. Los tres registros de `1800.txt` reproducen el ejemplo visible de la diapositiva 3. Los otros seis registros son ficticios, creados para esta práctica. El resultado de los nueve registros corresponde a esta práctica, no a un conjunto completo mostrado en la presentación.

`FICHEROS` contiene los datos dentro del propio programa. Los archivos de `datos/` son su copia en UTF-8 con saltos de línea LF. No necesitas leerlos del disco para ejecutar el programa.

Cada fichero representa un split lógico didáctico. El recorrido se ejecutará de forma local y serial. H/M son las etiquetas de este conjunto didáctico. Contamos apariciones de registros, sin deduplicar personas entre ficheros.

## Trabajo paso a paso

1. Lee los nueve registros y calcula a mano el resultado esperado. Anota qué campo debe ser la clave intermedia para poder sumar por categoría.
2. Completa `map_registro`. Separa los campos y comprueba que hay un nombre no vacío, una sola coma y una categoría válida. Emite un par por registro. Pistas: `rpartition` y `strip`.
3. Completa `shuffle`. Recorre los pares emitidos y construye una lista de valores por clave. Conserva todos los valores repetidos. Pista: `defaultdict`.
4. Completa `reduce_categoria`. Escribe un bucle que acumule los valores de un grupo y devuelve el par de salida. Después contrasta tu acumulación con `sum`.
5. Completa `ejecutar_mapreduce`. Escribe los bucles que recorren los ficheros y sus líneas, recoge las emisiones, agrupa y reduce. Si un registro falla, informa del fichero y la línea. Pistas: `splitlines` y `enumerate`.
6. Completa `conteo_secuencial` con un recorrido directo que actualice los contadores sin construir emisiones ni grupos.
7. Completa `mostrar_demostracion`: muestra las nueve emisiones Map, los grupos y los dos resultados Reduce. Compara los métodos y comprueba que no se pierde ningún registro.

## Evidencias que debes obtener

- Nueve emisiones Map, una por registro.
- Dos grupos, uno por categoría, con todos sus valores.
- Un diccionario final con el conteo que hayas calculado para cada categoría.
- El mismo resultado mediante el conteo secuencial.
- Igualdad entre registros, emisiones Map y valores agrupados. Comprueba que `H + M = 9` usando los conteos que hayas obtenido.

Un registro vacío, uno sin coma, uno sin nombre o uno con categoría distinta de H/M debe provocar un error claro. No lo ignores silenciosamente. Los saltos LF que cierran los registros no forman un registro adicional.

## Relación con la presentación

Las diapositivas 7–9 usan el nombre como clave intermedia. La diapositiva 10 compara después la clave con H/M. Explica por qué esas dos decisiones no encajan. Para este objetivo la clave intermedia es la categoría, que es precisamente aquello por lo que se quiere contar.

## Qué representa la simulación

Las funciones muestran la transformación Map, la agrupación por clave y la reducción. El programa no ejecuta Hadoop ni reproduce HDFS, la transferencia por red, collect, spill, merge, planificación o tolerancia a fallos. No se deben extraer conclusiones de rendimiento distribuido a partir de esta ejecución local.


## Código 1 Presentación del bloque


In [ ]:
"""Plantilla intermedia: conteo de registros por categoría H/M.

Python 3.10 o posterior. Solo biblioteca estándar.
Completa las funciones sustituyendo cada NotImplementedError.
Los tres primeros registros son los del ejemplo de la diapositiva 3.
Los otros seis son ficticios y se han añadido para esta práctica.
Cada fichero representa un split lógico didáctico. La ejecución es serial
y local. Esta plantilla no ejecuta Hadoop.
"""


## Código 2 Imports y datos preparados


In [2]:
from collections import defaultdict


FICHEROS: dict[str, str] = {
    "1800.txt": (
        "Manuel García García, H\n"
        "Juan Belmonte Pérez, H\n"
        "María Selas Rodríguez, M\n"
    ),
    "1801.txt": (
        "Lucía Martín Pérez, M\n"
        "Diego Ramos Díaz, H\n"
        "Elena Suárez León, M\n"
    ),
    "1802.txt": (
        "Pablo Torres Gil, H\n"
        "Nora Medina Ruiz, M\n"
        "Iván Cabrera Soto, H\n"
    ),
}

CATEGORIAS = ("H", "M")


## Código 3 Map: parseo, validación y emisión


In [3]:
def map_registro(linea: str) -> tuple[str, int]:
    """Recibe una línea sin terminador y emite (categoría, 1).

    TODO: separa nombre y categoría. Rechaza nombres vacíos, registros
    sin una única coma, saltos de línea internos y categorías inválidas.
    Lee H/M del campo final. No deduzcas la categoría a partir del nombre.
    Pista: investiga rpartition y strip.
    """
    # 1. Validación de saltos de línea
    if "\n" in linea or "\r" in linea:
        raise ValueError("La línea contiene saltos de línea internos")
    
    # 2. Separar por comas
    nombre_raw, sep, categoria_raw = linea.rpartition(",")

    # 3. Comprobar que solo hay una coma
    if sep != ",":
        raise ValueError("La línea no contiene exactamente una coma")
    if "," in nombre_raw:
        raise ValueError("El nombre contiene más de una coma")
    
    # 4. Limpiar espacios y validar contenido
    nombre = nombre_raw.strip()
    categoria = categoria_raw.strip()

    # 5. Validar nombre y categoría
    if not nombre:
        raise ValueError("El nombre está vacío")
    if categoria not in CATEGORIAS:
        raise ValueError(f"Categoría inválida: {categoria}")

    # 6. Emitir par (categoría, 1)
    return (categoria, 1)


## Código 4 Shuffle: agrupación sin perder repeticiones


In [4]:
def shuffle(pares: list[tuple[str, int]]) -> dict[str, list[int]]:
    """Recibe emisiones Map y devuelve una lista de valores por clave.

    TODO: escribe el recorrido de pares y la construcción de los grupos.
    Cada 1 debe conservarse, aunque otros valores sean iguales.
    Pista: defaultdict puede facilitar la creación de listas vacías.
    """
    
    grupos: dict[str, list[int]] = defaultdict(list)
    for clave, valor in pares:
        grupos[clave].append(valor)
    return dict(grupos)

## Código 5 Reduce: cálculo por categoría


In [5]:
def reduce_categoria(clave: str, valores: list[int]) -> tuple[str, int]:
    """Devuelve (categoría, total) para un único grupo.

    TODO: escribe la acumulación de los valores y devuelve el par de salida.
    Pista: resuélvelo primero con un bucle. Después puedes compararlo con sum.
    """
    total = 0
    for valor in valores:
        total += valor
    assert total == sum(valores), "La suma manual no coincide con sum()"
    return (clave, total)


## Código 6 Coordinación de las fases


In [6]:
def ejecutar_mapreduce(ficheros: dict[str, str]) -> dict[str, int]:
    """Devuelve el conteo de las categorías presentes en los registros.

    TODO: recorre los ficheros y sus líneas, llama a Map y recoge sus pares.
    Después agrupa y aplica Reduce a cada clave. Construye el resultado.
    Los errores deben indicar fichero y número de línea, sin ignorarlos.
    Pista: splitlines, enumerate y los contratos de las funciones anteriores.
    """
    emisiones: list[tuple[str, int]] = []

    # 1. Fase Map: recorrer cada fichero y cada línea
    for nombre_fichero, contenido in ficheros.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            try:
                par = map_registro(linea)
                emisiones.append(par)
            except ValueError as err:
                raise ValueError(
                    f"Error en '{nombre_fichero}', línea {num_linea} ('{linea}'): {err}"
                ) from err

    # 2. Fase Shuffle: agrupar por clave
    grupos = shuffle(emisiones)

    # 3. Fase Reduce: sumar cada grupo y construir el diccionario final
    resultado: dict[str, int] = {}
    for clave, valores in grupos.items():
        categoria, total = reduce_categoria(clave, valores)
        resultado[categoria] = total

    return resultado


## Código 7 Contraste secuencial


In [7]:
def conteo_secuencial(ficheros: dict[str, str]) -> dict[str, int]:
    """Cuenta directamente los registros H/M sin crear listas de emisiones.

    TODO: recorre y valida los datos. Actualiza un contador por categoría.
    Este recorrido permitirá contrastar tu resultado MapReduce.
    """
    contadores: dict[str, int] = {}

    for nombre_fichero, contenido in ficheros.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            try:
                categoria, _ = map_registro(linea)
            except ValueError as err:
                raise ValueError(
                    f"Error en '{nombre_fichero}', línea {num_linea} ('{linea}'): {err}"
                ) from err

            contadores[categoria] = contadores.get(categoria, 0) + 1

    return contadores


## Código 8 Mostrar las fases y comprobar el resultado


In [8]:
def mostrar_demostracion() -> None:
    """Construye una traza de los datos preparados.

    TODO: muestra una emisión por registro, los grupos y cada salida Reduce.
    Ejecuta ambos métodos y comprueba su igualdad. Comprueba también que el
    número de registros, emisiones y valores agrupados coincide con H + M.
    Calcula los conteos de H y M. Su suma debe ser igual a los 9 registros.
    """
    print("=== 1. FASE MAP (Emisiones por registro) ===")
    emisiones: list[tuple[str, int]] = []
    total_registros = 0

    for nombre_fichero, contenido in FICHEROS.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            total_registros += 1
            try:
                par = map_registro(linea)
                emisiones.append(par)
                print(f"  [{nombre_fichero}:{num_linea}] '{linea}' -> {par}")
            except ValueError as err:
                raise ValueError(
                    f"Error en '{nombre_fichero}', línea {num_linea} ('{linea}'): {err}"
                ) from err

    print("\n=== 2. FASE SHUFFLE (Agrupación por categoría) ===")
    grupos = shuffle(emisiones)
    for clave, valores in grupos.items():
        print(f"  Grupo '{clave}': {valores}")

    print("\n=== 3. FASE REDUCE (Resultados por categoría) ===")
    resultado_traza: dict[str, int] = {}
    for clave, valores in grupos.items():
        par_reducido = reduce_categoria(clave, valores)
        resultado_traza[par_reducido[0]] = par_reducido[1]
        print(f"  Reduce '{clave}' -> {par_reducido}")

    print("\n=== 4. COMPARACIÓN Y COMPROBACIONES ===")
    resultado_mr = ejecutar_mapreduce(FICHEROS)
    resultado_sec = conteo_secuencial(FICHEROS)

    print(f"  Resultado MapReduce : {resultado_mr}")
    print(f"  Resultado Secuencial: {resultado_sec}")

    # Evidencias requeridas por el guion
    total_valores_agrupados = sum(len(v) for v in grupos.values())
    conteo_h = resultado_mr.get("H", 0)
    conteo_m = resultado_mr.get("M", 0)
    suma_hm = conteo_h + conteo_m

    assert len(emisiones) == 9, "Debe haber exactamente 9 emisiones Map."
    assert len(grupos) == 2, "Debe haber exactamente 2 grupos (H y M)."
    assert resultado_mr == resultado_traza == resultado_sec, (
        "Los resultados de MapReduce y el conteo secuencial no coinciden."
    )
    assert total_registros == len(emisiones) == total_valores_agrupados == suma_hm == 9, (
        "Se ha perdido algún registro durante el proceso."
    )

    print(
        f"  Comprobación OK: Registros ({total_registros}) == "
        f"Emisiones ({len(emisiones)}) == Valores agrupados ({total_valores_agrupados}) == "
        f"H + M ({conteo_h} + {conteo_m} = {suma_hm})."
    )


## Código 9 Ejecutar tu demostración


In [9]:
if __name__ == "__main__":
    mostrar_demostracion()


=== 1. FASE MAP (Emisiones por registro) ===
  [1800.txt:1] 'Manuel García García, H' -> ('H', 1)
  [1800.txt:2] 'Juan Belmonte Pérez, H' -> ('H', 1)
  [1800.txt:3] 'María Selas Rodríguez, M' -> ('M', 1)
  [1801.txt:1] 'Lucía Martín Pérez, M' -> ('M', 1)
  [1801.txt:2] 'Diego Ramos Díaz, H' -> ('H', 1)
  [1801.txt:3] 'Elena Suárez León, M' -> ('M', 1)
  [1802.txt:1] 'Pablo Torres Gil, H' -> ('H', 1)
  [1802.txt:2] 'Nora Medina Ruiz, M' -> ('M', 1)
  [1802.txt:3] 'Iván Cabrera Soto, H' -> ('H', 1)

=== 2. FASE SHUFFLE (Agrupación por categoría) ===
  Grupo 'H': [1, 1, 1, 1, 1]
  Grupo 'M': [1, 1, 1, 1]

=== 3. FASE REDUCE (Resultados por categoría) ===
  Reduce 'H' -> ('H', 5)
  Reduce 'M' -> ('M', 4)

=== 4. COMPARACIÓN Y COMPROBACIONES ===
  Resultado MapReduce : {'H': 5, 'M': 4}
  Resultado Secuencial: {'H': 5, 'M': 4}
  Comprobación OK: Registros (9) == Emisiones (9) == Valores agrupados (9) == H + M (5 + 4 = 9).


## Explicación de tus resultados

1. Lee los nueve registros y calcula a mano el resultado esperado. Anota qué campo debe ser la clave intermedia para poder sumar por categoría.
Deberían haber 5 H y 4 M. La clave debe ser el género, es decir M o H.
En la fase 1 mostramos todos los resultados individualmente, con su nombre y género.
En la segunda fase los dividimos por género pero no los sumamos.
En la fase 3 hacemos lo mismo que en la 2 pero sumando los resultados.
En la última fase mostramos el resultado mediante mapreduce y el secuencial.

### Relación con la presentación
En las diapositivas 7 a 9 se usa el nombre de la persona para agrupar los datos y el género (H o M) se deja guardado aparte dentro de los valores. Por eso, en la diapositiva 10 el programa falla al comprobar si la clave es H o M: lo que recibe en ese punto es un nombre como "Manuel" o "María". Como nuestro objetivo es contar registros por categoría, en Map debemos usar directamente la letra H o M como clave (junto al valor 1) para que se formen solo dos grupos y Reduce pueda sumarlos fácilmente.